# Harmony Project 3 — LoRA Fine-Tuning (Qwen2.5-7B)



## Cell 1 — Install pinned dependencies

`transformers==4.46.3` + `trl==0.11.4` is the only combination that works with the multi-GPU LoRA flow used in Cell 6.

In [1]:
!pip install -q \
  "transformers==4.46.3" \
  "trl==0.11.4" \
  peft bitsandbytes accelerate datasets wandb json-repair

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.

## Cell 2 — Configuration

Only edit `BASE_MODEL_REVISION` if a newer commit on HF Hub is desired. Everything else is locked.

In [2]:
BASE_MODEL           = "Qwen/Qwen2.5-7B-Instruct"
BASE_MODEL_REVISION  = "a09a35458c702b33eeacc393d103063234e8bc28"
WANDB_PROJECT        = "harmony-p3-lora"
SEED                 = 42

TRAIN_PATH = "/kaggle/input/datasets/b35keertanakscs105/harmony-ade-data/train.jsonl"
VAL_PATH   = "/kaggle/input/datasets/b35keertanakscs105/harmony-ade-data/val.jsonl"

print("Config loaded.")

Config loaded.


## Cell 3 — Pre-flight checks

Verifies GPU, data files, row counts, and JSONL format. If any assert fails, **stop and fix before running Cell 6**.

In [3]:
import os, json, torch
from pathlib import Path

# 1. GPU
assert torch.cuda.is_available(), "No GPU — this notebook requires T4×2"
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")

# 2. Data files
assert Path(TRAIN_PATH).exists(), f"MISSING: {TRAIN_PATH}"
assert Path(VAL_PATH).exists(),   f"MISSING: {VAL_PATH}"

# 3. Row counts
n_train = sum(1 for _ in open(TRAIN_PATH))
n_val   = sum(1 for _ in open(VAL_PATH))
print(f"Train rows: {n_train:,}   Val rows: {n_val:,}")
assert n_train > 15000, f"train.jsonl too small ({n_train})"
assert n_val   > 2000,  f"val.jsonl too small ({n_val})"

# 4. Format spot-check
with open(TRAIN_PATH) as f:
    s = json.loads(f.readline())
assert "messages" in s and "schema_version" in s["messages"][1]["content"]
print("✓ Pre-flight passed")

GPUs: 2
  GPU 0: Tesla T4  15.6 GB
  GPU 1: Tesla T4  15.6 GB
Train rows: 19,040   Val rows: 2,379
✓ Pre-flight passed


## Cell 4 — Load tokenizer ONLY

**Do not load the model here.** Cell 6 loads it with a balanced GPU split. Loading it here would put 14 GB on one GPU, breaking Cell 6.

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"✓ Tokenizer loaded  (vocab={len(tokenizer):,})")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✓ Tokenizer loaded  (vocab=151,665)


## Cell 5 — Load and format dataset

Applies the Qwen2.5 chat template to produce a `text` field for SFTTrainer. Token length max is ~565 after templating — `max_seq_length=600` in Cell 6 handles this with no truncation.

In [5]:
import json
import numpy as np
from datasets import Dataset

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

def format_example(ex):
    return {
        "text": tokenizer.apply_chat_template(
            ex["messages"], tokenize=False, add_generation_prompt=False
        )
    }

train_dataset = Dataset.from_list(load_jsonl(TRAIN_PATH)).map(format_example)
val_dataset   = Dataset.from_list(load_jsonl(VAL_PATH)).map(format_example)
print(f"Train: {len(train_dataset):,}   Val: {len(val_dataset):,}")

# Quick token-length check (sample of 200)
lens = [len(tokenizer(ex["text"], add_special_tokens=False)["input_ids"])
        for ex in train_dataset.select(range(200))]
print(f"Token length  p50={np.percentile(lens,50):.0f}  "
      f"p95={np.percentile(lens,95):.0f}  max={max(lens)}")
assert max(lens) <= 1024, f"Some examples > 1024 tokens: {max(lens)}"
print("✓ Dataset ready")

Map:   0%|          | 0/19040 [00:00<?, ? examples/s]

Map:   0%|          | 0/2379 [00:00<?, ? examples/s]

Train: 19,040   Val: 2,379
Token length  p50=451  p95=552  max=565
✓ Dataset ready


## Cell 6 — Single 1-epoch run (Option B — fits in one Kaggle session)

Runs one training job with hardcoded lr=2e-4, rank=16. Completes in ~5–6 hours.
Saves adapter to `/kaggle/working/adapter/` when done. Download before session ends.

In [6]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 6 — SINGLE 1-EPOCH FINAL RUN  (Option B — fits in one Kaggle session)
# Hardcoded: lr=2e-4, rank=16 (well-known safe defaults for LoRA on Qwen2.5)
# ════════════════════════════════════════════════════════════════════════════
import os, gc, json, torch
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
from kaggle_secrets import UserSecretsClient
from transformers import AutoModelForCausalLM
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig
import wandb

# ── PATCH: fix multi-GPU device mismatch in transformers 4.46.3 loss ────────
import transformers.loss.loss_utils as _loss_utils

def _patched_xent(source, target, num_items_in_batch=None, ignore_index=-100, **kwargs):
    reduction = "sum" if num_items_in_batch is not None else "mean"
    loss = torch.nn.functional.cross_entropy(
        source, target, ignore_index=ignore_index, reduction=reduction)
    if reduction == "sum":
        if hasattr(num_items_in_batch, "to"):
            num_items_in_batch = num_items_in_batch.to(loss.device)
        loss = loss / num_items_in_batch
    return loss

_loss_utils.fixed_cross_entropy = _patched_xent
print("✓ Patched fixed_cross_entropy for multi-GPU")
# ────────────────────────────────────────────────────────────────────────────

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["WANDB_SILENT"] = "true"

# ── Safety: free anything left in VRAM ──────────────────────────────────────
for _v in ["model", "trainer"]:
    if _v in globals():
        del globals()[_v]
gc.collect(); torch.cuda.empty_cache()
print(f"Start  →  GPU 0: {torch.cuda.mem_get_info(0)[0]/1e9:.1f} GB free  |  "
      f"GPU 1: {torch.cuda.mem_get_info(1)[0]/1e9:.1f} GB free")
assert torch.cuda.mem_get_info(0)[0] > 14e9, (
    "GPU 0 is not empty — restart the kernel and re-run from Cell 1."
)
assert torch.cuda.mem_get_info(1)[0] > 14e9, (
    "GPU 1 is not empty — restart the kernel and re-run from Cell 1."
)

# ── Custom DataCollator ─────────────────────────────────────────────────────
class DataCollatorForCompletionOnlyLM:
    def __init__(self, response_template, tokenizer):
        self.tokenizer = tokenizer
        self.response_template_ids = tokenizer.encode(
            response_template, add_special_tokens=False)

    def __call__(self, instances):
        input_ids = [torch.tensor(inst["input_ids"]) for inst in instances]
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True,
            padding_value=self.tokenizer.pad_token_id)
        labels = input_ids.clone()
        tpl = self.response_template_ids
        for i in range(len(labels)):
            seq = labels[i].tolist()
            found = False
            for j in range(len(seq) - len(tpl) + 1):
                if seq[j: j + len(tpl)] == tpl:
                    labels[i, : j + len(tpl)] = -100
                    found = True
                    break
            if not found:
                labels[i, :] = -100
        labels[input_ids == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids": input_ids,
            "attention_mask": (input_ids != self.tokenizer.pad_token_id).long(),
            "labels": labels,
        }

# ── Hardcoded hyperparameters (Option B) ────────────────────────────────────
LEARNING_RATE = 2e-4
LORA_RANK     = 16
LORA_ALPHA    = LORA_RANK * 2
NUM_EPOCHS    = 1
RUN_NAME      = "lora_quickfinal_acct2"
OUT_DIR       = f"/kaggle/working/checkpoints/{RUN_NAME}"
ADAPTER_DIR   = "/kaggle/working/adapter"

print(f"\n{'='*62}")
print(f"  SINGLE RUN : {RUN_NAME}")
print(f"  lr={LEARNING_RATE}  rank={LORA_RANK}  epochs={NUM_EPOCHS}")
print(f"{'='*62}")

# ── Load model — balanced across both T4s ───────────────────────────────────
gc.collect(); torch.cuda.empty_cache()
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    revision=BASE_MODEL_REVISION,
    torch_dtype=torch.float16,
    device_map="balanced",
    max_memory={0: "10GiB", 1: "10GiB"},
)
model.config.use_cache = False
print(f"  After load  →  GPU 0: {torch.cuda.mem_get_info(0)[0]/1e9:.1f} GB free  |  "
      f"GPU 1: {torch.cuda.mem_get_info(1)[0]/1e9:.1f} GB free")

# ── LoRA config ─────────────────────────────────────────────────────────────
lora_cfg = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
)

collator = DataCollatorForCompletionOnlyLM(
    response_template="<|im_start|>assistant\n",
    tokenizer=tokenizer,
)

# ── W&B init ────────────────────────────────────────────────────────────────
wandb.init(
    project=WANDB_PROJECT,
    name=RUN_NAME,
    reinit=True,
    config={
        "lr": LEARNING_RATE, "rank": LORA_RANK, "alpha": LORA_ALPHA,
        "num_epochs": NUM_EPOCHS, "base_model": BASE_MODEL,
        "revision": BASE_MODEL_REVISION, "approach": "option_b_quickfinal",
    },
)

# ── Auto-resume if checkpoint exists ────────────────────────────────────────
resume_ckpt = None
if Path(OUT_DIR).exists():
    ckpts = sorted(Path(OUT_DIR).glob("checkpoint-*"),
                   key=lambda p: int(p.name.split("-")[-1]))
    if ckpts:
        resume_ckpt = str(ckpts[-1])
        print(f"  ↩  Resuming from {resume_ckpt}")

# ── SFT config ──────────────────────────────────────────────────────────────
sft_config = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,
    average_tokens_across_devices=False,
    fp16=True, bf16=False,
    max_seq_length=600,
    dataset_text_field="text",
    eval_strategy="steps", eval_steps=200,
    save_strategy="steps", save_steps=200, save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=20,
    report_to="wandb", run_name=RUN_NAME,
    seed=SEED, data_seed=SEED,
)

# ── Train ───────────────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model, args=sft_config,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    data_collator=collator, peft_config=lora_cfg,
    tokenizer=tokenizer,
)

trainer.train(resume_from_checkpoint=resume_ckpt)
best_eval_loss = trainer.state.best_metric
print(f"\n  ✓  Training complete  →  best_eval_loss = {best_eval_loss:.4f}")

# ── Save adapter + tokenizer + metadata ─────────────────────────────────────
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
meta = {
    "run_name": RUN_NAME,
    "approach": "option_b_quickfinal",
    "base_model": BASE_MODEL,
    "base_model_revision": BASE_MODEL_REVISION,
    "learning_rate": LEARNING_RATE,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "best_eval_loss": round(float(best_eval_loss), 6),
    "num_epochs": NUM_EPOCHS,
    "max_seq_length": 600,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "effective_batch_size": 16,
    "optimizer": "adamw_torch",
    "fp16": True, "bf16": False,
    "target_modules": ["q_proj","k_proj","v_proj","o_proj",
                       "gate_proj","up_proj","down_proj"],
    "lora_dropout": 0.05,
}
with open(f"{ADAPTER_DIR}/training_args.json", "w") as f:
    json.dump(meta, f, indent=2)

wandb.finish()
print(f"\n{'='*62}")
print(f"  DONE — Adapter saved → {ADAPTER_DIR}")
print(f"  Final eval_loss = {best_eval_loss:.4f}")
print(f"{'='*62}")
print("\n⚠  Download /kaggle/working/adapter/ as a zip before session ends.")

2026-05-28 17:44:53.099685: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779990293.515504      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779990293.632229      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779990294.577098      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779990294.577143      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779990294.577146      23 computation_placer.cc:177] computation placer alr

✓ Patched fixed_cross_entropy for multi-GPU
Start  →  GPU 0: 15.5 GB free  |  GPU 1: 15.5 GB free

  SINGLE RUN : lora_quickfinal_acct2
  lr=0.0002  rank=16  epochs=1


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

...

Map:   0%|          | 0/19040 [00:00<?, ? examples/s]

Map:   0%|          | 0/2379 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


Step,Training Loss,Validation Loss
200,0.042700,0.017656
400,0.038400,0.017132
600,0.032100,0.017330
800,0.028100,0.012182
1000,0.022500,0.011455



  ✓  Training complete  →  best_eval_loss = 0.0115

  DONE — Adapter saved → /kaggle/working/adapter
  Final eval_loss = 0.0115

⚠  Download /kaggle/working/adapter/ as a zip before session ends.


## Done

When Cell 6 finishes:

1. **Download** `/kaggle/working/adapter/` as a zip (right-click → Download)
2. **Note** the W&B run URL (visit https://wandb.ai/your_username/harmony-p3-lora)
3. **Commit** the adapter zip to `models/adapters/lora_v1/` in the Harmony repo

If the kernel hits the 12-hour limit:
- Checkpoint is in `/kaggle/working/checkpoints/lora_quickfinal_acct2/`
- Re-run cells 1 → 6 and Cell 6 will auto-resume from the last checkpoint